<center>
    <p style="text-align:center">
    <img alt="arize logo" src="https://storage.googleapis.com/arize-assets/arize-logo-white.jpg" width="300"/>
        <br>
        <a href="https://docs.arize.com/arize/">Docs</a>
        |
        <a href="https://github.com/Arize-ai">GitHub</a>
    </p>
</center>

# Build, Test, and Optimize a Trip-Planner Prompt

An end-to-end walkthrough of the prompt iteration cycle in Arize AX, using a trip-planner use case.

You'll work through three sections that mirror the [Prompts concept docs](https://arize.com/docs/ax/concepts/prompts/overview):

1. **Create** — define a Prompt Object and save it to Prompt Hub.
1. **Test** — run the prompt against a dataset with an evaluator, as an Arize experiment.
1. **Optimize** — iterate the prompt, save a new version, and compare runs side by side.

By the end you'll have a versioned trip-planner prompt in Prompt Hub, two experiment runs against the same dataset, and a sense of where each version did better. Total cost: a few cents on `gpt-4o-mini`.

## Setup

Install dependencies, load API keys, and initialize the Arize client.

In [1]:
!pip install -qq "arize>=8.0.0" openai pandas

In [2]:
import os
from getpass import getpass

ARIZE_API_KEY = os.environ.get("ARIZE_API_KEY") or getpass("🔑 Enter your Arize API key: ")
ARIZE_SPACE_ID = os.environ.get("ARIZE_SPACE_ID") or getpass("🔑 Enter your Arize space ID: ")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY") or getpass("🔑 Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [3]:
from arize.client import ArizeClient
from uuid import uuid4

client = ArizeClient(api_key=ARIZE_API_KEY)

# Unique suffix so this notebook can run cleanly each time without name collisions.
RUN_ID = uuid4().hex[:8]
print(f"Run ID: {RUN_ID}")

Run ID: b2b53ca3


## Section 1 — Create the prompt

A trip planner takes a destination, duration, travel style, and some research/budget/local context, and produces a structured day-by-day itinerary.

Our **Prompt Object** has five parts:

- **Prompt template** — system + user messages with `{variable}` placeholders.
- **Model** — `gpt-4o-mini`.
- **Invocation parameters** — temperature, max tokens, etc.
- **Tools** — none for this prompt.
- **Response format** — plain text in this version (we'll tighten this later).

See [The Prompt Object](https://arize.com/docs/ax/concepts/prompts/prompt-object) for the conceptual breakdown.

In [4]:
from arize._generated.api_client.models.input_variable_format import InputVariableFormat
from arize._generated.api_client.models.llm_provider import LlmProvider
from arize._generated.api_client.models.llm_message import LLMMessage
from arize._generated.api_client.models.invocation_params import InvocationParams

PROMPT_NAME = f"trip-planner-{RUN_ID}"

system_message_v1 = (
    "You are a trip planner. Given a destination, duration, travel style, "
    "and supplementary research, produce a day-by-day itinerary."
)

user_message = (
    "{duration} itinerary for {destination} in {travel_style} style.\n\n"
    "Research: {research}\n"
    "Budget: {budget_info}\n"
    "Local notes: {local_info}\n\n"
    "Format: Day X: Time - Activity - Cost"
)

prompt_v1 = client.prompts.create(
    space=ARIZE_SPACE_ID,
    name=PROMPT_NAME,
    description="Day-by-day itinerary generator for a given destination, duration, and travel style.",
    commit_message="v1: baseline trip planner",
    input_variable_format=InputVariableFormat.F_STRING,
    provider=LlmProvider.OPEN_AI,
    model="gpt-4o-mini",
    messages=[
        LLMMessage(role="system", content=system_message_v1),
        LLMMessage(role="user", content=user_message),
    ],
    invocation_params=InvocationParams(temperature=0.7, max_completion_tokens=600),
)

print(f"Created prompt {prompt_v1.name} (id={prompt_v1.id})")
print(f"Initial version: {prompt_v1.version.id}")

  arize.pre_releases | WARNING | [BETA] prompts.create is a beta API in Arize SDK v8.32.1 and may change without notice. If you experience unexpected failures, please upgrade to the most recent version of the package.


Created prompt trip-planner-b2b53ca3 (id=UHJvbXB0OjM0ODYwOkxDZTE=)
Initial version: UHJvbXB0VmVyc2lvbjo4OTc3NjpvYWxn


## Section 2 — Test the prompt against a dataset

Now we need data. We'll define a small dataset of trip-planner inputs covering different destinations and travel styles, upload it to Arize, and run the prompt against it as an experiment.

An **experiment** combines:

- A **dataset** of input rows.
- A **task** — here, rendering the prompt and calling the LLM.
- One or more **evaluators** that score the output.

See [Experiments for prompts](https://arize.com/docs/ax/concepts/prompts/experiments-for-prompts) for the concept.

In [5]:
import pandas as pd

DATASET_NAME = f"trip-planner-test-set-{RUN_ID}"

examples = [
    {
        "destination": "Istanbul, Turkey",
        "duration": "3 days",
        "travel_style": "standard",
        "research": "Best time to visit is April-May. Top attractions: Hagia Sophia, Blue Mosque, Grand Bazaar.",
        "budget_info": "Mid-range hotels $80-120/night, meals $15-30 each, transport $5-10/day.",
        "local_info": "Try Karaköy for breakfast spots; avoid Sultanahmet for dinner due to tourist markup.",
    },
    {
        "destination": "Bangkok, Thailand",
        "duration": "4 days",
        "travel_style": "family-friendly",
        "research": "Cool season Nov-Feb best. Top kid-friendly: Safari World, Dream World, river cruise.",
        "budget_info": "Family rooms $100-180/night, street food $2-5, taxis $5-15.",
        "local_info": "Skytrain reliable; visit temples early morning to beat heat.",
    },
    {
        "destination": "Barcelona, Spain",
        "duration": "5 days",
        "travel_style": "romantic",
        "research": "Spring/fall ideal. Sagrada Familia, Park Güell, tapas tours, beach access from city center.",
        "budget_info": "Boutique hotels $150-250/night, tapas dinner $30-50, metro $12 day pass.",
        "local_info": "Dinner starts late (9pm+); reserve Sagrada Familia tickets weeks ahead.",
    },
    {
        "destination": "Reykjavik, Iceland",
        "duration": "4 days",
        "travel_style": "adventure",
        "research": "Sept-Mar for northern lights; June-Aug for midnight sun. Blue Lagoon, Golden Circle, glacier tours.",
        "budget_info": "Hotels $200-300/night, dinner $40-70, day tours $80-150.",
        "local_info": "Rent a car for flexibility; weather changes fast — pack layers.",
    },
    {
        "destination": "New York City, USA",
        "duration": "2 days",
        "travel_style": "standard",
        "research": "Times Square, Central Park, museums (Met, MoMA), Brooklyn Bridge walk.",
        "budget_info": "Hotels $250-400/night, meals $20-50, subway $33 7-day pass.",
        "local_info": "Walk where possible; subway faster than taxis in midtown.",
    },
]

examples_df = pd.DataFrame(examples)
dataset = client.datasets.create(
    name=DATASET_NAME,
    space=ARIZE_SPACE_ID,
    examples=examples_df,
)
print(f"Created dataset {dataset.name} with {len(examples)} rows (id={dataset.id})")

  arize.pre_releases | WARNING | [BETA] datasets.create is a beta API in Arize SDK v8.32.1 and may change without notice. If you experience unexpected failures, please upgrade to the most recent version of the package.


Created dataset trip-planner-test-set-b2b53ca3 with 5 rows (id=RGF0YXNldDozNDk1Njg6b3cwSQ==)


### Define the task

The **task** is what runs per row. For a prompt experiment, the task renders the prompt's template with the row's variables, calls the model, and returns the text output.

In production you'd fetch the prompt by `name` + `label` (e.g., `production`); for this notebook we render the messages inline so the cell is self-contained.

In [6]:
from openai import OpenAI

oai = OpenAI()

def make_task(system_message: str, user_template: str, model: str = "gpt-4o-mini"):
    """Build an experiment task that runs the trip-planner prompt against each row."""

    def task(dataset_row) -> str:
        rendered_user = user_template.format(**dataset_row)
        resp = oai.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_message},
                {"role": "user", "content": rendered_user},
            ],
            temperature=0.7,
            max_tokens=600,
        )
        return resp.choices[0].message.content

    return task

task_v1 = make_task(system_message_v1, user_message)

### Define the evaluator

We want to know if the prompt produces a well-structured itinerary. A simple evaluator returns a label, a numeric score, and an explanation per row.

Here we check two structural properties: that the output mentions every day in the requested duration, and that it includes time-activity-cost markers. A real eval would use [LLM-as-a-judge](https://arize.com/docs/ax/concepts/evaluators/online-llm-as-judge) on subjective quality — we keep this one deterministic so it's reproducible in a notebook.

In [7]:
import re
from arize.experiments.types import EvaluationResult

def itinerary_structure_eval(output, dataset_row) -> EvaluationResult:
    """Score: 1.0 if every day is present AND the output looks like time-activity-cost rows."""
    try:
        if not output or not isinstance(output, str):
            return EvaluationResult(label="empty", score=0.0, explanation="Empty or non-string output.")

        # dataset_row may be a dict-like ReadOnly view — normalize to dict access.
        row = dataset_row if isinstance(dataset_row, dict) else dict(dataset_row)
        duration = str(row.get("duration", ""))
        match = re.search(r"\d+", duration)
        n_days = int(match.group()) if match else 1

        days_present = sum(
            1 for d in range(1, n_days + 1)
            if re.search(rf"\bDay\s*{d}\b", output, re.IGNORECASE)
        )
        days_coverage = days_present / n_days

        # Look for cost markers on at least half the days
        cost_markers = len(re.findall(r"\$|USD|EUR|GBP", output))
        has_costs = cost_markers >= max(1, n_days // 2)

        score = days_coverage * (1.0 if has_costs else 0.5)
        label = "well_structured" if score >= 0.9 else ("partial" if score >= 0.5 else "poor")

        return EvaluationResult(
            label=label,
            score=round(float(score), 2),
            explanation=(
                f"{days_present}/{n_days} days present; "
                f"{'has' if has_costs else 'lacks'} per-day cost markers."
            ),
        )
    except Exception as e:
        return EvaluationResult(label="error", score=0.0, explanation=f"Eval crashed: {e}")

### Run the first experiment

`client.experiments.run(...)` runs the task against every dataset row, scores each output with the evaluator, and returns a dataframe of results.

In [8]:
experiment_v1, results_v1 = client.experiments.run(
    name=f"trip-planner-v1-{RUN_ID}",
    dataset=dataset.name,
    space=ARIZE_SPACE_ID,
    task=task_v1,
    evaluators={"itinerary_structure": itinerary_structure_eval},
    concurrency=3,
    exit_on_error=True,
)

print(f"Columns: {list(results_v1.columns)}")
print()
print("Per-row scores (v1):")
# Find the eval columns by suffix — actual column name format may vary by SDK version.
score_col = next((c for c in results_v1.columns if c.endswith("score") and "itinerary" in c), None)
label_col = next((c for c in results_v1.columns if c.endswith("label") and "itinerary" in c), None)
if score_col and label_col:
    print(results_v1[[score_col, label_col]].to_string())
    print(f"\nMean score: {results_v1[score_col].mean():.2f}")
else:
    print(results_v1.head().to_string())

  arize.experiments.functions | INFO | 🧪 Experiment started.


  arize.experiments.evaluators.executors | WARNING | 🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.


running tasks |          | 0/5 (0.0%) | ⏳ 00:00<? | ?it/s

  arize.experiments.functions | INFO | ✅ Task runs completed.
Tasks Summary (06/08/26 02:05 PM -0700)
---------------------------------------
   n_examples  n_runs  n_errors
0           5       5         0


  arize.experiments.evaluators.executors | WARNING | 🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.


running experiment evaluations |          | 0/5 (0.0%) | ⏳ 00:00<? | ?it/s

  arize.experiments.functions | INFO | ✅ All evaluators completed.


  arize.pre_releases | WARNING | [BETA] experiments.get is a beta API in Arize SDK v8.32.1 and may change without notice. If you experience unexpected failures, please upgrade to the most recent version of the package.


Columns: ['example_id', 'output', 'error', 'result.trace.id', 'result.trace.timestamp', 'eval.itinerary_structure.score', 'eval.itinerary_structure.label', 'eval.itinerary_structure.explanation', 'eval.itinerary_structure.trace.id', 'eval.itinerary_structure.trace.timestamp']

Per-row scores (v1):
   eval.itinerary_structure.score eval.itinerary_structure.label
0                            1.00                well_structured
1                            0.75                        partial
2                            0.80                        partial
3                            0.75                        partial
4                            1.00                well_structured

Mean score: 0.86


## Section 3 — Iterate and compare

Looking at v1's outputs, suppose we want the itineraries to be more disciplined — always include cost markers, always use the `Day X: HH:MM - Activity - $cost` shape. Let's tighten the system prompt and save it as a new immutable version of the same Prompt Object.

See [Versioning and tags](https://arize.com/docs/ax/concepts/prompts/versioning-and-tags) for why each save creates a new hashed version.

In [9]:
system_message_v2 = (
    "You are a disciplined trip planner. For every day of the trip you MUST output:\n"
    "  Day N: HH:MM - Activity - $cost\n"
    "Use one bullet per time slot. ALWAYS include a $cost figure (estimate if exact "
    "price unknown). Cover the full duration. Be concise; no preamble or epilogue."
)

prompt_v2 = client.prompts.create_version(
    prompt=prompt_v1.id,
    space=ARIZE_SPACE_ID,
    commit_message="v2: enforce strict day/time/cost format, drop preamble",
    input_variable_format=InputVariableFormat.F_STRING,
    provider=LlmProvider.OPEN_AI,
    model="gpt-4o-mini",
    messages=[
        LLMMessage(role="system", content=system_message_v2),
        LLMMessage(role="user", content=user_message),
    ],
    invocation_params=InvocationParams(temperature=0.5, max_completion_tokens=600),
)

print(f"Saved v2 (version id: {prompt_v2.id})")

  arize.pre_releases | WARNING | [BETA] prompts.create_version is a beta API in Arize SDK v8.32.1 and may change without notice. If you experience unexpected failures, please upgrade to the most recent version of the package.


Saved v2 (version id: UHJvbXB0VmVyc2lvbjo4OTc3NzpiVDRl)


In [10]:
# Tag v2 as production. The tag is mutable — next iteration's v3 can take it later.
client.prompts.set_labels(version_id=prompt_v2.id, labels=["production"])
print("v2 tagged as production.")

  arize.pre_releases | WARNING | [BETA] prompts.set_labels is a beta API in Arize SDK v8.32.1 and may change without notice. If you experience unexpected failures, please upgrade to the most recent version of the package.


v2 tagged as production.


In [11]:
task_v2 = make_task(system_message_v2, user_message)

experiment_v2, results_v2 = client.experiments.run(
    name=f"trip-planner-v2-{RUN_ID}",
    dataset=dataset.name,
    space=ARIZE_SPACE_ID,
    task=task_v2,
    evaluators={"itinerary_structure": itinerary_structure_eval},
    concurrency=3,
    exit_on_error=True,
)

print("Per-row scores (v2):")
score_col_v2 = next((c for c in results_v2.columns if c.endswith("score") and "itinerary" in c), None)
label_col_v2 = next((c for c in results_v2.columns if c.endswith("label") and "itinerary" in c), None)
if score_col_v2 and label_col_v2:
    print(results_v2[[score_col_v2, label_col_v2]].to_string())
    print(f"\nMean score: {results_v2[score_col_v2].mean():.2f}")
else:
    print(results_v2.head().to_string())

  arize.experiments.functions | INFO | 🧪 Experiment started.


  arize.experiments.evaluators.executors | WARNING | 🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.


running tasks |          | 0/5 (0.0%) | ⏳ 00:00<? | ?it/s

  arize.experiments.functions | INFO | ✅ Task runs completed.
Tasks Summary (06/08/26 02:06 PM -0700)
---------------------------------------
   n_examples  n_runs  n_errors
0           5       5         0


  arize.experiments.evaluators.executors | WARNING | 🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.


running experiment evaluations |          | 0/5 (0.0%) | ⏳ 00:00<? | ?it/s

  arize.experiments.functions | INFO | ✅ All evaluators completed.


Per-row scores (v2):
   eval.itinerary_structure.score eval.itinerary_structure.label
0                             1.0                well_structured
1                             1.0                well_structured
2                             1.0                well_structured
3                             1.0                well_structured
4                             1.0                well_structured

Mean score: 1.00


In [12]:
# Compare per-row
score_col = next(c for c in results_v1.columns if c.endswith("score") and "itinerary" in c)
delta = pd.DataFrame({
    "v1_score": results_v1[score_col].values,
    "v2_score": results_v2[score_col].values,
})
delta["delta"] = delta["v2_score"] - delta["v1_score"]
print(delta.to_string())
print(f"\nMean v1: {delta['v1_score'].mean():.2f}")
print(f"Mean v2: {delta['v2_score'].mean():.2f}")

   v1_score  v2_score  delta
0      1.00       1.0   0.00
1      0.75       1.0   0.25
2      0.80       1.0   0.20
3      0.75       1.0   0.25
4      1.00       1.0   0.00

Mean v1: 0.86
Mean v2: 1.00


## Next steps

You've created a Prompt Object, run two versions through experiments against the same dataset, and compared their scores.

Where to go from here:

- **Open the prompt in Prompt Hub** to see both versions side by side, diff the templates, and tag one as `production`. Your application can fetch by tag via `client.prompts.get(prompt=name, label="production")` — see [Loading prompts in your application](https://arize.com/docs/ax/concepts/prompts/loading-in-applications).
- **Run the experiments view** to see both runs side by side, complete with per-row outputs. The Compare Experiments mode shows you exactly which rows v2 changed.
- **Add an LLM-as-a-judge evaluator** for the subjective dimensions (writing quality, usefulness) the deterministic eval can't capture. See [Evaluators](https://arize.com/docs/ax/concepts/evaluators/overview).
- **Wire into CI/CD** so a prompt edit becomes a PR check. See [Prompts in CI/CD](https://arize.com/docs/ax/concepts/prompts/prompts-in-ci-cd).
- **Try Prompt Learning** for automated optimization once you have a golden dataset. See [Optimizing prompts](https://arize.com/docs/ax/concepts/prompts/optimizing-prompts).